# 03 wazuh event processing

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
from pathlib import Path
import pandas as pd
import json
import gzip

PROJECT_ROOT = PUBLICATION_ROOT

WAZUH_DIR = PROJECT_ROOT / "data" / "wazuh" / "logs"

# Find archived and active JSON logs
archive_files = sorted(WAZUH_DIR.rglob("*.json.gz"))
json_files = sorted(WAZUH_DIR.rglob("*.json"))

print("Compressed JSON files:", len(archive_files))
print("Uncompressed JSON files:", len(json_files))

print("\nFirst five compressed files:")
for file in archive_files[:5]:
    print(file.relative_to(WAZUH_DIR))

print("\nFirst five uncompressed files:")
for file in json_files[:5]:
    print(file.relative_to(WAZUH_DIR))

In [ ]:
# Locate only raw Wazuh archive files
ARCHIVES_DIR = WAZUH_DIR / "archives"

raw_files = sorted(
    list(ARCHIVES_DIR.rglob("*.json.gz")) +
    list(ARCHIVES_DIR.rglob("*.json"))
)

print("Raw archive files:", len(raw_files))

# Read the first JSON event from the first archive
file_path = raw_files[0]

if file_path.name.endswith(".gz"):
    opener = gzip.open
else:
    opener = open

with opener(file_path, "rt", encoding="utf-8") as file:
    first_event = json.loads(file.readline())

print("\nSource file:", file_path.name)

# Display the JSON in a readable format
print(json.dumps(first_event, indent=2))

In [ ]:
# Access the nested Windows event information
win_data = first_event.get("data", {}).get("win", {})

system = win_data.get("system", {})
event_data = win_data.get("eventdata", {})

# Extract important fields
event_summary = {
    "WazuhTimestamp": first_event.get("timestamp"),
    "Agent": first_event.get("agent", {}).get("name"),
    "Provider": system.get("providerName"),
    "EventID": system.get("eventID"),
    "WindowsEventTime": system.get("systemTime"),
    "Computer": system.get("computer"),
}

print("EVENT SUMMARY")
for key, value in event_summary.items():
    print(f"{key}: {value}")

print("\nEVENT-SPECIFIC FIELDS")
for key, value in event_data.items():
    print(f"{key}: {value}")

In [ ]:
from collections import Counter

event_counts = Counter()
total_events = 0

for file_path in raw_files:
    opener = gzip.open if file_path.suffix == ".gz" else open

    with opener(file_path, "rt", encoding="utf-8-sig") as file:
        for line in file:
            if not line.strip():
                continue

            event = json.loads(line)

            system = event.get("data", {}).get("win", {}).get("system", {})

            provider = system.get("providerName")
            event_id = system.get("eventID")

            if provider and event_id:
                event_counts[(provider, str(event_id))] += 1
                total_events += 1

# Convert counts into a DataFrame
event_summary = pd.DataFrame(
    [
        {
            "Provider": provider,
            "EventID": event_id,
            "Count": count
        }
        for (provider, event_id), count in event_counts.items()
    ]
)

event_summary = event_summary.sort_values(
    "Count", ascending=False
)

print("Total Windows/Sysmon events:", total_events)

display(event_summary.head(20))

In [ ]:
# Extract Windows Security and Sysmon events from Wazuh archives

import csv

OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "wazuh_relevant_events.csv"

# Match the reference pipeline's provider/channel policy
allowed_sources = {
    ("Microsoft-Windows-Security-Auditing", "Security"),
    ("Microsoft-Windows-Sysmon", "Microsoft-Windows-Sysmon/Operational")
}

fieldnames = [
    "WazuhTimestamp",
    "WindowsEventTime",
    "Agent",
    "Provider",
    "EventChannel",
    "EventID",
    "EventRecordID",
    "Computer",
    "Keywords",
    "SeverityValue",
    "EventData"
]

extracted_count = 0

with open(output_file, "w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    writer.writeheader()

    for file_path in raw_files:
        opener = gzip.open if file_path.suffix == ".gz" else open

        with opener(file_path, "rt", encoding="utf-8-sig") as source:
            for line in source:
                if not line.strip():
                    continue

                event = json.loads(line)

                win = event.get("data", {}).get("win", {})
                system = win.get("system", {})

                provider = system.get("providerName", "")
                channel = system.get("channel", "")

                # Retain supported Windows event sources
                if (provider, channel) not in allowed_sources:
                    continue

                writer.writerow({
                    "WazuhTimestamp": event.get("timestamp"),
                    "WindowsEventTime": system.get("systemTime"),
                    "Agent": event.get("agent", {}).get("name"),
                    "Provider": provider,
                    "EventChannel": channel,
                    "EventID": str(system.get("eventID", "")),
                    "EventRecordID": system.get("eventRecordID"),
                    "Computer": system.get("computer"),
                    "Keywords": system.get("keywords"),
                    "SeverityValue": system.get("severityValue"),
                    "EventData": json.dumps(win.get("eventdata", {}))
                })

                extracted_count += 1

print("Extracted events:", extracted_count)
print("Saved to:", output_file)

In [ ]:
# Read a sample of the extracted events
events_sample = pd.read_csv(
    output_file,
    nrows=1000
)

print("Sample shape:", events_sample.shape)

print("\nColumn names:")
print(events_sample.columns.tolist())

print("\nEvent distribution in sample:")
print(
    events_sample["EventID"].value_counts()
)

display(events_sample.head(5))

In [ ]:
# Check potential duplicates in the first 1,000 records
potential_duplicates = events_sample.duplicated(
    subset=["Agent", "Provider", "EventRecordID"],
    keep=False
)

print("Potential duplicate records:", potential_duplicates.sum())

display(
    events_sample.loc[
        potential_duplicates,
        ["Agent", "Provider", "EventID", "EventRecordID"]
    ].head(10)
)

In [ ]:
system = first_event.get("data", {}).get("win", {}).get("system", {})

print("Event channel:", system.get("channel"))
print("Event record ID:", system.get("eventRecordID"))
print("Computer:", system.get("computer"))
print("Provider:", system.get("providerName"))

In [ ]:
# Deduplicate extracted Windows events in chunks
import csv
import pandas as pd

input_file = OUTPUT_DIR / "wazuh_relevant_events.csv"
dedup_file = OUTPUT_DIR / "wazuh_deduplicated_events.csv"

seen = set()
duplicate_count = 0
missing_key_count = 0
original_count = 0
unique_count = 0

with open(dedup_file, "w", newline="", encoding="utf-8") as output:
    writer = None

    for chunk in pd.read_csv(input_file, chunksize=50000, dtype=str):

        if writer is None:
            writer = csv.DictWriter(output, fieldnames=chunk.columns.tolist())
            writer.writeheader()

        for record in chunk.to_dict("records"):
            original_count += 1

            key_fields = [
                record["Computer"],
                record["EventChannel"],
                record["EventRecordID"]
            ]

            if any(pd.isna(value) or not value for value in key_fields):
                missing_key_count += 1
            else:
                key = tuple(key_fields)

                if key in seen:
                    duplicate_count += 1
                    continue

                seen.add(key)

            writer.writerow(record)
            unique_count += 1

print("Original events:", original_count)
print("Duplicate events removed:", duplicate_count)
print("Records with incomplete keys retained:", missing_key_count)
print("Remaining events:", unique_count)
print("Saved to:", dedup_file)

In [ ]:
import hashlib

seen_events = {}
identical_duplicates = 0
conflicting_duplicates = 0
conflict_examples = []

columns = [
    "Computer", "EventChannel", "EventRecordID",
    "WindowsEventTime", "EventID", "EventData"
]

for chunk in pd.read_csv(
    OUTPUT_DIR / "wazuh_relevant_events.csv",
    usecols=columns,
    dtype=str,
    chunksize=50000
):
    for record in chunk.to_dict("records"):
        key = (
            record["Computer"],
            record["EventChannel"],
            record["EventRecordID"]
        )

        content = (
            record["WindowsEventTime"],
            record["EventID"],
            record["EventData"]
        )

        fingerprint = hashlib.sha256(
            repr(content).encode("utf-8")
        ).digest()

        if key in seen_events:
            if seen_events[key] == fingerprint:
                identical_duplicates += 1
            else:
                conflicting_duplicates += 1

                if len(conflict_examples) < 5:
                    conflict_examples.append(key)
        else:
            seen_events[key] = fingerprint

print("Identical duplicates:", identical_duplicates)
print("Conflicting duplicate identities:", conflicting_duplicates)
print("Example conflicts:", conflict_examples)

In [ ]:
# Extract structured event fields from deduplicated Wazuh telemetry

import csv
import json
import pandas as pd

input_file = OUTPUT_DIR / "wazuh_deduplicated_events.csv"
structured_file = OUTPUT_DIR / "wazuh_structured_events.csv"

fieldnames = [
    "WazuhTimestamp", "WindowsEventTime",
    "Agent", "Computer", "EventChannel",
    "Provider", "EventID", "EventRecordID",
    "Keywords", "SeverityValue",
    "Username", "SubjectDomainName",
    "SubjectUserSid", "SubjectLogonId",
    "TargetUsername", "TargetDomainName",
    "TargetUserSid", "TargetLogonId",
    "ShareName", "ShareLocalPath",
    "RelativeTargetName", "ObjectName", "ObjectType",
    "AccessMask", "HandleId", "Status",
    "Image", "CommandLine",
    "ProcessGuid", "ParentProcessGuid", "ProcessId",
    "TargetFilename", "DestinationIp",
    "DestinationPort", "Initiated"
]

processed_count = 0

with open(structured_file, "w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    writer.writeheader()

    for chunk in pd.read_csv(input_file, chunksize=50000, dtype=str):
        for record in chunk.to_dict("records"):

            details = json.loads(record["EventData"])

            writer.writerow({
                "WazuhTimestamp": record["WazuhTimestamp"],
                "WindowsEventTime": record["WindowsEventTime"],
                "Agent": record["Agent"],
                "Computer": record["Computer"],
                "EventChannel": record["EventChannel"],
                "Provider": record["Provider"],
                "EventID": record["EventID"],
                "EventRecordID": record["EventRecordID"],
                "Keywords": record["Keywords"],
                "SeverityValue": record["SeverityValue"],

                "Username": details.get(
                    "subjectUserName", details.get("user")
                ),
                "SubjectDomainName": details.get("subjectDomainName"),
                "SubjectUserSid": details.get("subjectUserSid"),
                "SubjectLogonId": details.get("subjectLogonId"),

                "TargetUsername": details.get("targetUserName"),
                "TargetDomainName": details.get("targetDomainName"),
                "TargetUserSid": details.get("targetUserSid"),
                "TargetLogonId": details.get("targetLogonId"),

                "ShareName": details.get("shareName"),
                "ShareLocalPath": details.get("shareLocalPath"),
                "RelativeTargetName": details.get("relativeTargetName"),
                "ObjectName": details.get("objectName"),
                "ObjectType": details.get("objectType"),
                "AccessMask": details.get("accessMask"),
                "HandleId": details.get("handleId"),
                "Status": details.get("status"),

                "Image": details.get("image"),
                "CommandLine": details.get("commandLine"),
                "ProcessGuid": details.get("processGuid"),
                "ParentProcessGuid": details.get("parentProcessGuid"),
                "ProcessId": details.get("processId"),
                "TargetFilename": details.get("targetFilename"),

                "DestinationIp": details.get("destinationIp"),
                "DestinationPort": details.get("destinationPort"),
                "Initiated": details.get("initiated")
            })

            processed_count += 1

print("Structured events:", processed_count)
print("Saved to:", structured_file)